## Initial Setup
**Before Starting**, open Terminal and run `az login`, then follow the prompts to ensure you are authenticated to Azure. Then run the next cell to load the necessary libraries.


**Important:** Before beginning, you need to install `ollama` and run `ollama pull` to retrieve a model that ollama will run, then start ollama via `litellm --model ollama/MODELNAME`  

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core.tools import FunctionTool
from autogen_core.models import ModelInfo
from autogen_ext.models.openai import OpenAIChatCompletionClient,AzureOpenAIChatCompletionClient
from autogen_agentchat.teams import MagenticOneGroupChat
from autogen_ext.models.ollama import OllamaChatCompletionClient
from datetime import datetime
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider, AzureCliCredential
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

## Configure Clients and Deployments
Next, you need to initialize an instance of `AzureOpenAIChatCompletionClient()`. It is strongly recommend you review the documentation for Autogen, but briefly, you need to provide a model name, an azure endpoint, a deployment name, and an API version.

    AzureOpenAIChatCompletionClient(
        model,
        azure_endpoint,
        azure_deployment,
        api_version,
        azure_ad_token_provider
    )

**Note:** If using a model client through `ollama`, use this exactly the same as `AzureOpenAIChatCompletionClient()`


In [ ]:
# examples

# leave this unchanged
token_provider = get_bearer_token_provider(AzureCliCredential(), "https://cognitiveservices.azure.com/.default")

# define a model client for gpt_4o
model_client_gpt=AzureOpenAIChatCompletionClient(
    model="gpt-4o-2024-08-06",
    azure_endpoint="YOUR ENDPOINT",
    azure_deployment="gpt-4o",
    api_version="2024-05-01-preview",
    azure_ad_token_provider=token_provider,
)

# define model client for local LLM served by ollama
ollama_client=OllamaChatCompletionClient(
    model="mixtral:8x22b-instruct",
)


## Setup Agents
Next, you must define the agents that go into the team for Round Robin or MagenticOne using the `AssistantAgent()` class.

    AssistantAgent(
        name,
        model_client,
        description,
        system_message
    )

The arguments are described below:

* `name` -> a unique human-readable name for the agent, do not use spaces or special characters
* `model_client` -> assign one of the model_clients that you created above here
* `description` -> the documentation is a bit vague on this, but we've found empirically that you should always include a relevant and applicable value here, both because this is provided in the prompt to the LLM and because changing it will impact the response from the LLM
* `system_message` -> the system message that is provided to the LLM


The next cell has examples.

In [ ]:
# example function that returns three agents: moderator, up, down

# functions
def defineAgents(note_text, model_client_gpt):
    sys_message_moderatorAgent = f"""You are a moderator, helping a team of doctors review a CLINICAL_SUMMARY of some CLINICAL_NOTES.
    Your goal is to get all of the doctors to come to a consensus about their final rating of the CLINICIAL_SUMMARY, using the RUBRIC_SET.
    You must let the doctors talk, have a chance to respond to each other, and reach a consensus.
    Once consensus is reached, end the discussion by saying CONSENSUS and repeating the final categories.
    ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
    sys_message_moderatorAgent += "Here is the note text:\n"
    sys_message_moderatorAgent += note_text
    moderator_agent = AssistantAgent(
        name = "moderator",
        model_client = model_client_gpt,
        description = "An agent focused on moderating the discussion amongst a team to reach a consensus, but always lets others have a chance to contribute.",
        system_message = sys_message_moderatorAgent
    )
    
    sys_message_downAgent = f"""You are a Primary Care Doctor.
        YOU TEND TO SUGGEST CATEGORIES ARE TOO HIGH, AND SHOULD BE LOWER, WITH SUPPORTING EVIDENCE.
        Be professional with your response.
        IF ARGUMENTS ARE STRONG ENOUGH IN THE OPPOSING DIRECTION, YOU ARE WILLING TO BUDGE.
        ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
    sys_message_downAgent += "Here is the note text:\n"
    sys_message_downAgent += note_text
    down_agent = AssistantAgent(
        name="down",
        model_client = model_client_gpt,
        description = "An agent that prioritizes pessimistic evaluations always in favor of being harsher when it comes to scoring.",
        system_message = sys_message_downAgent
    )
    
    sys_message_upAgent = f"""You are a Primary Care Doctor.
        YOU SHOULD GENERALLY ARGUE THAT SUGGESTED CATEGORIES ARE TOO LOW, AND SHOULD BE HIGHER, WITH SUPPORTING EVIDENCE.
        Be professional with your response.
        IF ARGUMENTS ARE STRONG ENOUGH IN THE OPPOSING DIRECTION, YOU ARE WILLING TO BUDGE.
        ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
    sys_message_upAgent += "Here is the note text:\n"
    sys_message_upAgent += note_text
    up_agent = AssistantAgent(
        name="up",
        model_client = model_client_gpt,
        description = "An agent that prioritizes optimistic evaluations always in favor of being lenient when it comes to scoring.",
        system_message = sys_message_upAgent
    )

    # return down_agent, up_agent
    return moderator_agent, down_agent, up_agent

## Configure MagenticOneOrchestrator or RoundRobin
Finally, you need to configure MagenticOne or RoundRobin (both are shown).

Similar to the agentic prompts, you need to design a prompt for the MagenticOne Orchestrator or a task for the RoundRobin.

The usage for either is to create a team (more on this in a moment), then call `team.run_stream(task)`

This is an asynchronous process (note that autogen is not thread-safe), so it must be called via the `async` directive. Additionally, you must use the `async` library if you run this outside of a jupyter notebook.

In [ ]:
async def assistant_run_stream(team, task_text) -> list:
    l = []
    async for message in team.run_stream(task=task_text):
        l.append(message)
    return l

In [ ]:
# MagenticOne prompt example

m1_task ="You are a summarization quality expert that specializes in text analysis and reasoning. "
m1_task += "Your task is to analyze multiple evaluations generated by different Large Language Model (LLM) agents that you will create "
m1_task += "for the same text input and follow the provided rubric and instructions to determine the final score for each attribute in the rubric. "
m1_task += "Indicate the start and end of each round of discussion with the LLM agents, and if discussion with the LLM Agents is ever reset. " 
m1_task += "Provide your reasoning when generating the final output.\n\n" 



## Team Setup
Finally, setup the team.

    # MagenticOne
    team = MagenticOneGroupChat(
        participants=[moderator_agent, down_agent, up_agent], termination_condition=termination, model_client=model_client_orchestrator, max_turns=6, max_stalls=2
    )

    # Round Robin
    termnination = TextMentionTermination("TERMINATE")
    team = RoundRobinGroupChat(
        participants=[moderator_agent, down_agent, up_agent], termination_condition=termination, model_client=model_client_orchestrator
    )

* Assign the team members as a list for `participants`
* The `termination_condition` is a condition that stops discussion. This can be including the text "TERMINATE" as one of the agents instructions or calling `termination=TerminationCondition()`
* `max_turns` can be set to limit the rounds of discussion
* `max_stalls` (MagenticOneGroupChat only) sets the number of times the orchestrator will step in and reset the conversation. If this happens the MagenticOneOrchestrator will usually have a sentence with the word "reset" in it.
  * It's a good idea to set `max_turns` and set a termination condition, particularly when calling `RoundRobinGroupChat`, to prevent the discussion from continuing too long (and racking up charges).
* Assign the model client you created to the `model_client` value


In [ ]:
# then call the function you created earlier 
agent_responses = await assistant_run_stream(team=team, task_text=task_text)

# if run from a script, call via 

async.run(assistant_run_stream(team=team, task_text=task_text)

# Alternatively you can use Console to get the output streamed to the notebook or to STDOUT from a script

await Console(team.run_stream(task=task))

